# Preprocessing

In [12]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection  import train_test_split

In [13]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.sample(2)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
376,901315,B,10.57,20.22,70.15,338.3,0.09073,0.1660,0.2280,0.05941,...,22.82,76.51,351.9,0.1143,0.3619,0.6030,0.1465,0.2597,0.12000,NaN
95,86208,M,20.26,23.03,132.40,1264.0,0.09078,0.1313,0.1465,0.08683,...,31.59,156.10,1750.0,0.1190,0.3539,0.4098,0.1573,0.3689,0.08368,NaN


In [14]:
df.shape

(569, 33)

In [15]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [16]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [17]:
# scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
# Label Encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [19]:
# Numpy arrays to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

# Define Model

In [33]:
class MySimpleNN():
  def __init__(self,X):
    self.weights = torch.rand(X.shape[1],1, dtype= torch.float64,requires_grad=True)
    self.bias = torch.zeros(1,dtype = torch.float64,requires_grad=True)

  def forward(self,X):
    z = torch.matmul(X,self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


# Important Parameter

In [50]:
learning_rate = 0.1
epochs = 25

# Training Pipeline

In [51]:
# create model
model = MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model.forward(X_train_tensor)

  # calculate the loss
  loss = model.loss_function(y_pred,y_train_tensor)
  print(f'epoch {epoch + 1} loss {loss}')

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

epoch 1 loss 3.6186376758896768
epoch 2 loss 3.4891840633002644
epoch 3 loss 3.3567961776173765
epoch 4 loss 3.219987112004713
epoch 5 loss 3.0809027171868175
epoch 6 loss 2.936953675734044
epoch 7 loss 2.7814847220921575
epoch 8 loss 2.6201525406489194
epoch 9 loss 2.4551257142688057
epoch 10 loss 2.2901068582261273
epoch 11 loss 2.1210089071546365
epoch 12 loss 1.9481721329783555
epoch 13 loss 1.776832335494413
epoch 14 loss 1.6109001395950668
epoch 15 loss 1.452799640260723
epoch 16 loss 1.3090842043423097
epoch 17 loss 1.1822801695299852
epoch 18 loss 1.0744992126437825
epoch 19 loss 0.9868961276100475
epoch 20 loss 0.919164829426342
epoch 21 loss 0.8693255642843651
epoch 22 loss 0.8340581018084421
epoch 23 loss 0.8095582041500756
epoch 24 loss 0.7924176947403662
epoch 25 loss 0.780057762751954


# Evaluation

In [52]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6206524968147278
